<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part F: Appendices</h2>
<h2>Notebook F01b: Preparing the CDC Dataset</h2>
</div>

This notebook downloads the CDC regional air temperature dataset from Hugging Face, saves it to the correct location in the repository, and runs a quick validation so you can be confident the file is ready to use.

Run it once before starting Parts A or B. You do not need to run it again unless you want to refresh the data.

---

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>1. Dataset Overview</h3>
</div>

The CDC dataset contains monthly regional air temperature measurements across Germany, sourced from the [Deutscher Wetterdienst (DWD)](https://www.dwd.de).

| File | Frequency | Used in |
|------|-----------|---------|
| `cdc_monthly_regional_air_temp_D.parquet` | Monthly | Parts A, B |

The file is hosted in the Hugging Face repository [`mt0rm0/cdcdata`](https://huggingface.co/datasets/mt0rm0/cdcdata) and will be saved to `data/raw/cdcdata/` inside the course repository.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>2. Download</h3>
</div>

We use the `huggingface_hub` library to download the file.

In [ ]:
from huggingface_hub import hf_hub_download

import nb_config
from nb_config import CDC_TEMP_PATH

HF_REPO = "mt0rm0/cdcdata"
HF_REPO_TYPE = "dataset"
FILENAME = "cdc_monthly_regional_air_temp_D.parquet"

CDC_TEMP_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Downloading {FILENAME} ...")
hf_hub_download(
    repo_id=HF_REPO,
    repo_type=HF_REPO_TYPE,
    filename=FILENAME,
    local_dir=CDC_TEMP_PATH.parent,
)
print(f"  Saved to: {CDC_TEMP_PATH}")
print("\nDownload complete.")

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>3. Light Cleaning</h3>
</div>

The raw file has a few things to sort out:

- The index is a meaningless default integer index. The actual dates are in a column called `datum`, already typed as `datetime64[ns]`, so we just set that as the index.
- All region columns come in as strings and need to be cast to float.
- Finally, we sort by the new index and save the cleaned file back to `data/raw/cdcdata/`.

In [ ]:
import pandas as pd

df = pd.read_parquet(CDC_TEMP_PATH)

# Set datum as the index — it is already datetime64[ns], no parsing needed
df = df.set_index("Datum")
df.index.name = "date"

# Cast region columns from string to float
df = df.apply(pd.to_numeric, errors="coerce")

df = df.sort_index()

df.to_parquet(CDC_TEMP_PATH)

print(f"Shape: {df.shape}")
print(f"Index: {df.index.dtype}  |  range: {df.index.min()} to {df.index.max()}")
print(f"Dtypes after cleaning:\n{df.dtypes}")
print("\nCleaning complete.")

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>4. Validation</h3>
</div>

A quick sanity check to confirm the file is present, readable, and has the expected structure. Every check should show a green tick before you move on.

In [ ]:
import pandas as pd

print("--- cdc_monthly_regional_air_temp_D ---")
issues = []

if not CDC_TEMP_PATH.exists():
    print(f"  ❌  File not found at {CDC_TEMP_PATH}")
else:
    print(f"  ✅  File exists")

    df = pd.read_parquet(CDC_TEMP_PATH)
    print(f"  ✅  Readable  |  shape: {df.shape}")

    if isinstance(df.index, pd.DatetimeIndex):
        print(f"  ✅  DatetimeIndex  |  {df.index.min()} to {df.index.max()}")
    else:
        issues.append("Index is not a DatetimeIndex")
        print(f"  ❌  Index is not a DatetimeIndex (got {type(df.index).__name__})")

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    if numeric_cols:
        print(f"  ✅  Numeric columns ({len(numeric_cols)}): {numeric_cols[:5]}{' ...' if len(numeric_cols) > 5 else ''}")
    else:
        issues.append("No numeric columns found")
        print(f"  ❌  No numeric columns found")

    missing_pct = df.isnull().mean().mean() * 100
    print(f"  {'✅' if missing_pct < 5 else '⚠️ '}  Missing values: {missing_pct:.2f}% overall")

    if not issues:
        print(f"  All checks passed.")

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>5. Quick Preview</h3>
</div>

A final look at the first few rows to confirm everything looks right.

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.read_parquet(CDC_TEMP_PATH)
display(df.head())
print(f"Shape: {df.shape}")

---

The CDC dataset is ready. You can now open any notebook in Parts A or B that uses it.